
# 01 · Bronze Layer Exploration
Validate raw NYC Taxi data from Azure Open Datasets via OneLake shortcut.

In [ ]:
%pip install -e .

In [ ]:
from src.common.config import get_settings
from src.fabric.lakehouse.data_sources import SOURCE_REGISTRY
from src.fabric.lakehouse.bronze_ingest import ingest_to_bronze

settings = get_settings()
config = SOURCE_REGISTRY['nyctaxi_yellow']

# Ingest a single month to validate schema and row counts.
rows = ingest_to_bronze(
    spark=spark,
    config=config,
    workspace_id=settings.fabric_workspace_id,
    lakehouse_name=settings.lakehouse_name,
    year=2023,
    month=1,
)
print(f'Ingested {rows:,} rows into bronze_nyctaxi_yellow')

In [ ]:
# Profile the Bronze table: null rates, value ranges.
bronze_path = (
    f'abfss://{settings.fabric_workspace_id}@onelake.dfs.fabric.microsoft.com/'
    f'{settings.lakehouse_name}.Lakehouse/Tables/bronze_nyctaxi_yellow'
)
df = spark.read.format('delta').load(bronze_path)
df.select('fareAmount', 'tripDistance', 'passengerCount', 'tipAmount').summary().show()